In [129]:
# ============================================================
# STEP 4A — DiCE Setup
# PyTorch Binary Classifier Compatibility
# ============================================================

import dice_ml
from dice_ml import Dice
from dice_ml.model_interfaces.pytorch_model import PyTorchModel
from dice_ml.constants import ModelTypes
import numpy as np
import pandas as pd
import torch

print("DiCE Setup & Compatibility Configuration")
print("=" * 60)

# ------------------------------------------------------------
# 1. Prepare DiCE reference data
# ------------------------------------------------------------

dice_training_data = X_train_processed.copy()
dice_training_data["Diabetes_binary"] = (
    y_train_processed.astype(int)
)

feature_names = X_train_processed.columns.tolist()

# All 21 inputs are numerical values fed directly into the DNN.
# Treating them as continuous avoids the categorical-string
# incompatibility with pandas 3.x.

dice_data = dice_ml.Data(
    dataframe=dice_training_data,
    continuous_features=feature_names,
    outcome_name="Diabetes_binary"
)

print("DiCE Data Interface created:")
print(f"  Total features: {len(feature_names)}")
print(f"  Training samples: {len(dice_training_data):,}")

# ------------------------------------------------------------
# 2. Compatible PyTorch Model Interface
# ------------------------------------------------------------

class CompatiblePyTorchModel(PyTorchModel):

    def get_output(
        self,
        input_instance,
        model_score=True,
        transform_data=False,
        out_tensor=False
    ):

        # Get original sigmoid probability P(Diabetes)
        out = super().get_output(
            input_instance,
            model_score=model_score,
            transform_data=transform_data,
            out_tensor=out_tensor
        )

        if out_tensor:
            # P(No Diabetes)
            p1 = out.view(-1, 1)
            p0 = 1.0 - p1

            # Return [P(No Diabetes), P(Diabetes)]
            return torch.cat([p0, p1], dim=1)

        else:
            p1 = np.asarray(out).reshape(-1, 1)
            p0 = 1.0 - p1

            # Return [P(No Diabetes), P(Diabetes)]
            return np.hstack([p0, p1])


# IMPORTANT:
# Use the trained PRD-compliant model.
model.eval()

dice_model = CompatiblePyTorchModel(
    model=model,
    model_path="",
    backend="PYT"
)

# Required by dice-ml 0.12
dice_model.model_type = ModelTypes.Classifier

# ------------------------------------------------------------
# 3. Create DiCE Random Explainer
# ------------------------------------------------------------

dice_explainer = Dice(
    dice_data,
    dice_model,
    method="random"
)

# ------------------------------------------------------------
# 4. PRD Immutable Features
# ------------------------------------------------------------

immutable_features = [
    "Age",
    "Sex",
    "Education"
]

features_to_vary = [
    feature
    for feature in feature_names
    if feature not in immutable_features
]

print("\nDiCE Explainer created successfully.")
print("Method:", "random")

print("\nPRD Immutable Features")
print("-" * 60)

for feature in immutable_features:
    print(
        f"{feature:12s}: "
        f"{feature in feature_names}"
    )

print(
    f"\nMutable features: {len(features_to_vary)}"
)

# ------------------------------------------------------------
# 5. Verify DiCE output format
# ------------------------------------------------------------

test_query = X_test_processed.iloc[[0]]

output_probs = dice_model.get_output(test_query)

print("\nDiCE Prediction Interface Verification")
print("-" * 60)

print("Output:", output_probs)
print("Shape:", output_probs.shape)
print("Expected shape:", "(1, 2)")

print(
    "\nProbability sum:",
    output_probs[0].sum()
)

print(
    "P(No Diabetes):",
    output_probs[0, 0]
)

print(
    "P(Diabetes):",
    output_probs[0, 1]
)

print("\nDiCE setup completed.")

DiCE Setup & Compatibility Configuration
DiCE Data Interface created:
  Total features: 21
  Training samples: 177,576

DiCE Explainer created successfully.
Method: random

PRD Immutable Features
------------------------------------------------------------
Age         : True
Sex         : True
Education   : True

Mutable features: 18

DiCE Prediction Interface Verification
------------------------------------------------------------
Output: [[0.48197353 0.5180265 ]]
Shape: (1, 2)
Expected shape: (1, 2)

Probability sum: 1.0
P(No Diabetes): 0.48197353
P(Diabetes): 0.5180265

DiCE setup completed.


In [130]:
# ============================================================
# STEP 4B — Generate Counterfactuals for Test Instance 529
# ============================================================

test_instance_id = 529

query_instance = X_test_processed.iloc[[test_instance_id]]

print("DiCE Counterfactual Generation")
print("=" * 60)
print(f"Test Instance ID: {test_instance_id}")
print(f"True Label: {y_test_processed[test_instance_id]}")
print(f"Original Diabetes Probability: {model.predict(query_instance.values.astype(np.float32), verbose=False)[0][0] if hasattr(model, 'predict') else dice_model.get_output(query_instance)[0,1]:.6f}")

dice_exp = dice_explainer.generate_counterfactuals(
    query_instance,
    total_CFs=3,
    desired_class="opposite",
    features_to_vary=features_to_vary,
    verbose=False
)

print("\nCounterfactual generation completed.")

DiCE Counterfactual Generation
Test Instance ID: 529
True Label: 0.0
Original Diabetes Probability: 0.501891


100%|██████████| 1/1 [00:00<00:00,  2.07it/s]


Counterfactual generation completed.


In [131]:
# ============================================================
# STEP 4C — Verify Generated Counterfactuals
# ============================================================

cf_df = dice_exp.cf_examples_list[0].final_cfs_df

print("DiCE Counterfactual Verification")
print("=" * 60)

print(f"Number of counterfactuals generated: {len(cf_df)}")
print(f"Number of features: {cf_df.shape[1]}")
print("\nCounterfactuals:")
display(cf_df)

print("\nImmutable Feature Check")
print("-" * 60)

for feature in immutable_features:
    original_value = query_instance.iloc[0][feature]
    cf_values = cf_df[feature].values

    unchanged = np.allclose(
        cf_values.astype(float),
        float(original_value)
    )

    print(
        f"{feature:12s}: "
        f"Original = {original_value}, "
        f"CF values = {cf_values}, "
        f"Unchanged = {unchanged}"
    )

DiCE Counterfactual Verification
Number of counterfactuals generated: 3
Number of features: 22

Counterfactuals:


,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary
0,0.4,0.0,1.0,-0.813622,0.0,0.0,0.0,0.8,1.0,0.0,0.0,1.0,0.0,5.0,-0.429938,2.953912,0.0,1.0,0.317448,5.0,4.0,0
1,1.0,0.0,1.0,-0.813622,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,5.0,1.901469,2.953912,0.0,1.0,0.317448,5.0,4.0,0
2,1.0,0.0,1.0,-0.813622,0.0,0.0,0.0,0.0,1.0,0.0,1.1,1.0,0.0,5.0,-0.122852,2.953912,0.0,1.0,0.317448,5.0,4.0,0



Immutable Feature Check
------------------------------------------------------------
Age         : Original = 0.31744812317038673, CF values = [0.31744812 0.31744812 0.31744812], Unchanged = True
Sex         : Original = 1.0, CF values = [1. 1. 1.], Unchanged = True
Education   : Original = 5.0, CF values = [5. 5. 5.], Unchanged = True


In [132]:
# ============================================================
# STEP 4D — Verify Model Predictions for Counterfactuals
# ============================================================

cf_features = cf_df[feature_names].copy()

# Get model probabilities
cf_output = dice_model.get_output(cf_features)

print("Counterfactual Prediction Verification")
print("=" * 60)

for i in range(len(cf_df)):
    p_no_diabetes = float(cf_output[i, 0])
    p_diabetes = float(cf_output[i, 1])
    
    print(f"\nCounterfactual {i + 1}")
    print("-" * 40)
    print(f"P(No Diabetes): {p_no_diabetes:.6f}")
    print(f"P(Diabetes):    {p_diabetes:.6f}")
    print(f"Predicted Class: {int(p_diabetes >= 0.5)}")
    print(f"DiCE Outcome:   {int(cf_df.iloc[i]['Diabetes_binary'])}")

print("\nVerification completed.")

Counterfactual Prediction Verification

Counterfactual 1
----------------------------------------
P(No Diabetes): 0.584685
P(Diabetes):    0.415315
Predicted Class: 0
DiCE Outcome:   0

Counterfactual 2
----------------------------------------
P(No Diabetes): 0.510042
P(Diabetes):    0.489958
Predicted Class: 0
DiCE Outcome:   0

Counterfactual 3
----------------------------------------
P(No Diabetes): 0.670744
P(Diabetes):    0.329256
Predicted Class: 0
DiCE Outcome:   0

Verification completed.


In [133]:
# ============================================================
# STEP 4E — Generate DiCE Counterfactuals for All 600 Instances
# ============================================================

import time
import pickle

print("DiCE Full 600-Instance Experiment")
print("=" * 60)

dice_outputs = {}

successful = 0
failed = 0

start_time = time.time()

for count, test_instance_id in enumerate(shap_instance_ids, start=1):

    query_instance = X_test_processed.iloc[[test_instance_id]]

    try:
        dice_exp = dice_explainer.generate_counterfactuals(
            query_instance,
            total_CFs=3,
            desired_class="opposite",
            features_to_vary=features_to_vary,
            verbose=False
        )

        cf_df_instance = dice_exp.cf_examples_list[0].final_cfs_df.copy()

        dice_outputs[int(test_instance_id)] = {
            "test_instance_id": int(test_instance_id),
            "sigma_squared": float(
                stratified_samples.loc[
                    stratified_samples["test_instance_id"] == test_instance_id,
                    "sigma_squared"
                ].iloc[0]
            ),
            "uncertainty_stratum": str(
                stratified_samples.loc[
                    stratified_samples["test_instance_id"] == test_instance_id,
                    "uncertainty_stratum"
                ].iloc[0]
            ),
            "true_label": int(y_test_processed[test_instance_id]),
            "counterfactuals": cf_df_instance
        }

        successful += 1

    except Exception as e:
        failed += 1
        print(
            f"\nFAILED — Instance {test_instance_id}: "
            f"{type(e).__name__}: {e}"
        )

    if count % 50 == 0:
        elapsed = time.time() - start_time
        print(
            f"Processed {count}/600 | "
            f"Successful: {successful} | "
            f"Failed: {failed} | "
            f"Time: {elapsed:.1f}s"
        )

elapsed = time.time() - start_time

print("\n" + "=" * 60)
print("DiCE experiment completed.")
print(f"Total instances: {len(shap_instance_ids)}")
print(f"Successful: {successful}")
print(f"Failed: {failed}")
print(f"Total time: {elapsed:.2f} seconds")

# Save results
with open("dice_outputs.pkl", "wb") as f:
    pickle.dump(dice_outputs, f)

print("\nSaved: dice_outputs.pkl")

DiCE Full 600-Instance Experiment


100%|██████████| 1/1 [00:00<00:00,  2.21it/s]


Processed 50/600 | Successful: 50 | Failed: 0 | Time: 40.9s


100%|██████████| 1/1 [00:00<00:00,  2.30it/s]


Processed 100/600 | Successful: 100 | Failed: 0 | Time: 75.5s


...

Processed 500/600 | Successful: 499 | Failed: 1 | Time: 577.1s


100%|██████████| 1/1 [00:00<00:00,  2.27it/s]


Processed 550/600 | Successful: 549 | Failed: 1 | Time: 612.1s


100%|██████████| 1/1 [00:00<00:00,  2.25it/s]

Processed 600/600 | Successful: 599 | Failed: 1 | Time: 634.5s

DiCE experiment completed.
Total instances: 600
Successful: 599
Failed: 1
Total time: 634.52 seconds

Saved: dice_outputs.pkl


In [5]:
# ============================================================
# STEP 4F — Diagnose Failed DiCE Instance
# ============================================================

failed_id = 6718

query_failed = X_test_processed.iloc[[failed_id]]

original_probs = dice_model.get_output(query_failed)

print("Failed DiCE Instance Diagnosis")
print("=" * 60)

print(f"Test Instance ID: {failed_id}")
print(f"True Label: {y_test_processed[failed_id]}")
print(f"P(No Diabetes): {original_probs[0, 0]:.6f}")
print(f"P(Diabetes):    {original_probs[0, 1]:.6f}")
print(f"Predicted Class: {int(original_probs[0, 1] >= 0.5)}")

print("\nUncertainty Information")
print("-" * 60)

failed_uncertainty = stratified_samples[
    stratified_samples["test_instance_id"] == failed_id
]

display(failed_uncertainty)

print("\nOriginal Feature Values")
print("-" * 60)

display(query_failed.T)

NameError: name 'X_test_processed' is not defined

In [7]:
import os

print("Files currently available in /mnt/data:")
print("=" * 60)

for filename in sorted(os.listdir("/mnt/data")):
    print(filename)

Files currently available in /mnt/data:


FileNotFoundError: [WinError 3] The system cannot find the path specified: '/mnt/data'